# Deep Picard Iteration for European Call Option Pricing

This notebook prices a European call option and computes future exposure profiles
using the **Deep Picard Iteration** method from the `dim_fbsde` library.

The FBSDE system under the risk-neutral measure Q is:

$$dX_t = r X_t \, dt + \sigma X_t \, dW_t^Q, \quad X_0 = S_0$$

$$-dY_t = -r Y_t \, dt - Z_t \, dW_t^Q, \quad Y_T = \max(X_T - K, 0)$$

where $Y_t = V(t, X_t)$ is the option price and $Z_t = \sigma X_t \Delta(t, X_t)$ is the scaled delta.

**Solver**: `UncoupledFBSDESolver` (Deep Picard Iteration, gradient-based Z method)  
**Equation**: `EuropeanCallEquation` (custom `FBSDE` subclass)

## 1. Setup and Imports

In [ ]:
# Install the dim_fbsde package if needed
# !pip install -e /path/to/dim-fbsde

import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.stats import norm
import time
import pandas as pd
import logging

# dim_fbsde imports
from dim_fbsde.solvers.uncoupled import UncoupledFBSDESolver
from dim_fbsde.nets.mlp import MLP
from dim_fbsde.config import SolverConfig, TrainingConfig

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Logging (shows Picard iteration progress)
logging.basicConfig(level=logging.INFO, format='%(message)s')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 2. EuropeanCallEquation — FBSDE Definition

Subclasses the `FBSDE` base class from `dim_fbsde`. Encodes:
- **Forward SDE**: GBM under Q with drift $rX_t$ and diffusion $\sigma X_t$
- **Driver**: $f = -rY_t$ (pure discounting under Q)
- **Terminal condition**: $g(X_T) = \max(X_T - K, 0)$
- **Analytical solutions**: Black-Scholes price and delta for validation

In [ ]:
import math
from typing import Optional, Union
from dim_fbsde.equations.base import FBSDE


class EuropeanCallEquation(FBSDE):
    """
    FBSDE characterization of a European call option under Black-Scholes.

    System (risk-neutral measure Q):
        Forward:  dX_t = r*X_t dt + sigma*X_t dW_t^Q,   X_0 = S0
        Backward: -dY_t = -r*Y_t dt - Z_t dW_t^Q
        Terminal: Y_T = max(X_T - K, 0)

    Analytical solution:
        Y_t = V_BS(t, X_t)              (Black-Scholes call price)
        Z_t = sigma * X_t * N(d1)       (diffusion-scaled delta)

    Args:
        S0    : Initial asset price.
        K     : Strike price.
        r     : Risk-free rate.
        sigma : Volatility.
        T     : Option maturity.
        device: Torch device.
        dtype : Torch dtype.
    """

    def __init__(self,
                 S0: float = 100.0,
                 K: float = 100.0,
                 r: float = 0.01,
                 sigma: float = 0.25,
                 T: float = 1.0,
                 device: Union[str, torch.device] = 'cpu',
                 dtype: torch.dtype = torch.float32):
        super().__init__(dim_x=1, dim_y=1, dim_w=1,
                         x0=torch.tensor([S0]),
                         device=device, dtype=dtype)
        self.K = K
        self.r = r
        self.sigma = sigma
        self.T = T

    def drift(self, t, x, y, z, **kwargs):
        """Risk-neutral GBM drift: r * X_t. Shape: [Batch, 1]"""
        return self.r * x

    def diffusion(self, t, x, y, z, **kwargs):
        """GBM diffusion: sigma * X_t. Shape: [Batch, 1, 1]"""
        return (self.sigma * x).unsqueeze(-1)

    def driver(self, t, x, y, z, **kwargs):
        """
        Black-Scholes driver under Q: f = -r * Y_t.
        Under Q the Brownian motion absorbs the market price of risk,
        so the driver reduces to pure discounting. Shape: [Batch, 1]
        """
        return -self.r * y

    def terminal_condition(self, x, **kwargs):
        """Call payoff: max(X_T - K, 0). Shape: [Batch, 1]"""
        return torch.clamp(x - self.K, min=0.0)

    def analytical_y(self, t, x, **kwargs):
        """Black-Scholes call price V(t, X_t). Shape: [Batch, 1]"""
        T = kwargs.get('T_terminal', self.T)
        tau = torch.clamp(T - t, min=1e-8)
        log_m = torch.log(x / self.K)
        s = self.sigma * torch.sqrt(tau)
        d1 = (log_m + (self.r + 0.5 * self.sigma**2) * tau) / s
        d2 = d1 - s
        price = x * self._ncdf(d1) - self.K * torch.exp(-self.r * tau) * self._ncdf(d2)
        # At expiry, fall back to payoff
        if isinstance(T - t, torch.Tensor):
            at_expiry = ((T - t) <= 1e-8)
            if at_expiry.any():
                price = torch.where(at_expiry, torch.clamp(x - self.K, min=0.0), price)
        return price

    def analytical_z(self, t, x, **kwargs):
        """Diffusion-scaled delta: Z_t = sigma * X_t * N(d1). Shape: [Batch, 1, 1]"""
        T = kwargs.get('T_terminal', self.T)
        tau = torch.clamp(T - t, min=1e-8)
        log_m = torch.log(x / self.K)
        s = self.sigma * torch.sqrt(tau)
        d1 = (log_m + (self.r + 0.5 * self.sigma**2) * tau) / s
        return (self.sigma * x * self._ncdf(d1)).unsqueeze(1)

    @staticmethod
    def _ncdf(x):
        """Standard normal CDF via erf (fully differentiable)."""
        return 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))

## 3. Problem Parameters

In [ ]:
# --- Option / Market Parameters ---
S0               = 100.0   # Initial stock price
K                = 100.0   # Strike (at-the-money)
r                = 0.01    # Risk-free rate (1%)
sigma            = 0.25    # Volatility (25%)
T                = 1.0     # Maturity (1 year)

# --- Solver / Discretisation Parameters ---
N                = 50      # Time steps  (keep modest for gradient Z method)
num_paths        = 4000    # Monte Carlo paths for training
picard_iters     = 8       # Picard fixed-point iterations

# --- Network Architecture ---
hidden_dims      = [32, 32, 32]   # Hidden layer widths

# --- Training Hyperparameters ---
batch_size       = 512
epochs_per_iter  = 10      # Gradient steps per Picard iteration
learning_rate    = 5e-4

# --- Evaluation ---
P                = 2048    # Paths for exposure simulation

print('Configuration:')
print(f'  S0={S0}, K={K}, r={r*100}%, sigma={sigma*100}%, T={T}')
print(f'  N={N} steps, {num_paths} training paths, {picard_iters} Picard iterations')
print(f'  Network hidden dims: {hidden_dims}')

## 4. Black-Scholes Reference Price

In [ ]:
def bs_call(S, K, r, sigma, T):
    """Black-Scholes call price."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

exact_price = bs_call(S0, K, r, sigma, T)
print(f'Black-Scholes Exact Price: ${exact_price:.4f}')

## 5. Instantiate the FBSDE Equation and Solver

In [ ]:
# --- Equation ---
equation = EuropeanCallEquation(S0=S0, K=K, r=r, sigma=sigma, T=T, device=device)
print(equation)

# --- Configs ---
solver_cfg = SolverConfig(
    T=T,
    N=N,
    num_paths=num_paths,
    picard_iterations=picard_iters,
    z_method='gradient',   # Z computed via automatic differentiation
    device=device
)

train_cfg = TrainingConfig(
    batch_size=batch_size,
    epochs=epochs_per_iter,
    learning_rate=learning_rate,
    verbose=True
)

# --- Neural Network for Y(t, X_t) ---
# Input: (t, X_t) -> dim 1+1=2. Output: Y_t -> dim 1.
input_dim = 1 + equation.dim_x   # = 2
nn_Y = MLP(input_dim=input_dim, output_dim=equation.dim_y, hidden_dims=hidden_dims)
print(f'\nNetwork: {nn_Y}')
print(f'Parameters: {sum(p.numel() for p in nn_Y.parameters()):,}')

# --- Solver (gradient Z method — no nn_Z needed) ---
solver = UncoupledFBSDESolver(equation, solver_cfg, train_cfg, nn_Y=nn_Y, nn_Z=None)

## 6. Solve — Deep Picard Iteration

In [ ]:
print('Starting Deep Picard Iteration...\n')
t0 = time.time()
solution = solver.solve()
elapsed = time.time() - t0

print(f'\nSolved in {elapsed:.1f}s')

## 7. Results — Option Price at t=0

In [ ]:
# Y[:, 0, 0] is the network estimate of Y_0 = V(0, S0) across all paths
Y_paths = solution['Y']          # shape [num_paths, N+1, 1]
X_paths = solution['X']          # shape [num_paths, N+1, 1]
time_grid = solution['time']     # shape [N+1]

deep_price = float(np.mean(Y_paths[:, 0, 0]))
error = abs(deep_price - exact_price)
rel_error = error / exact_price * 100

print(f"{'='*55}")
print('PRICING RESULTS')
print(f"{'='*55}")
print(f'Black-Scholes Exact:    ${exact_price:.6f}')
print(f'Deep Picard (Y_0 mean): ${deep_price:.6f}')
print(f'Absolute Error:         ${error:.6f}')
print(f'Relative Error:          {rel_error:.4f}%')
print(f'Training Time:           {elapsed:.1f}s')
print(f"{'='*55}")

## 8. Convergence History

In [ ]:
history = solution['history']
iterations  = [h['iteration'] for h in history]
errors      = [h['error'] for h in history]
mean_losses = [float(np.mean(h['y_loss'])) for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Picard fixed-point error
axes[0].semilogy(iterations, errors, 'b-o', linewidth=2, markersize=6)
axes[0].set_xlabel('Picard Iteration', fontsize=12)
axes[0].set_ylabel('RMSE between iterates', fontsize=12)
axes[0].set_title('Picard Fixed-Point Error', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Training loss per iteration
axes[1].semilogy(iterations, mean_losses, 'g-o', linewidth=2, markersize=6)
axes[1].set_xlabel('Picard Iteration', fontsize=12)
axes[1].set_ylabel('Mean MSE Loss', fontsize=12)
axes[1].set_title('Network Training Loss per Picard Iteration', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Pathwise Comparison — Deep Picard vs Black-Scholes

In [ ]:
n_plot = 5   # number of sample paths to display

# Compute analytical Y along each simulated X path for comparison
X_tensor  = torch.tensor(X_paths[:n_plot], dtype=torch.float32, device=device)  # [n_plot, N+1, 1]
t_tensor  = torch.tensor(time_grid, dtype=torch.float32, device=device)          # [N+1]

Y_exact_paths = np.zeros((n_plot, N + 1))
for i, t_val in enumerate(time_grid):
    t_i  = torch.full((n_plot, 1), t_val, device=device)
    x_i  = X_tensor[:, i, :]
    y_bs = equation.analytical_y(t_i, x_i, T_terminal=T)
    Y_exact_paths[:, i] = y_bs.squeeze(-1).cpu().numpy()

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.tab10.colors

for p in range(n_plot):
    ax.plot(time_grid, Y_paths[p, :, 0],
            color=colors[p], linewidth=1.8, alpha=0.9,
            label=f'Deep Picard path {p+1}')
    ax.plot(time_grid, Y_exact_paths[p],
            color=colors[p], linewidth=1.8, alpha=0.5,
            linestyle='--')

# Legend proxy for line styles
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], color='k', lw=2, label='Deep Picard (solid)'),
           Line2D([0], [0], color='k', lw=2, ls='--', label='Black-Scholes (dashed)')]
ax.legend(handles=handles, fontsize=11)

ax.set_xlabel('Time (years)', fontsize=12)
ax.set_ylabel('Option Value ($)', fontsize=12)
ax.set_title('Pathwise Option Value: Deep Picard vs Black-Scholes', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Simulate Exposure Profiles (EPE / ENE)

Re-use the trained `nn_Y` to evaluate $P$ fresh forward paths and compute
**Discounted Expected Positive Exposure (DEPE)** and **Discounted Expected Negative Exposure (DENE)**:

$$\text{DEPE}(t) = \mathbb{E}\left[e^{-rt} \max(Y_t, 0)\right], \qquad
  \text{DENE}(t) = \mathbb{E}\left[e^{-rt} \min(Y_t, 0)\right]$$

We spin up a fresh `UncoupledFBSDESolver` with `P` paths, simulate the forward
process via `_simulate_forward_process()`, then call `_update_iterates()` once
with the already-trained `nn_Y` — no retraining, pure evaluation.

In [ ]:
import copy

print(f'Simulating {P} exposure paths with trained nn_Y...')
torch.manual_seed(0)

# --- Fresh solver with P paths, same equation and time grid ---
solver_cfg_eval = SolverConfig(
    T=T, N=N, num_paths=P,
    picard_iterations=1,
    z_method='gradient',
    device=device
)
train_cfg_eval = TrainingConfig(
    batch_size=P, epochs=1,
    learning_rate=learning_rate, verbose=False
)

# Deep-copy nn_Y so the evaluation solver owns its own reference
nn_Y_eval = copy.deepcopy(nn_Y)

eval_solver = UncoupledFBSDESolver(
    equation, solver_cfg_eval, train_cfg_eval, nn_Y=nn_Y_eval, nn_Z=None
)

# Step 1: simulate P fresh GBM paths using the equation's drift/diffusion
eval_solver._simulate_forward_process()

# Step 2: initialise Y/Z tensors (required before calling _update_iterates)
eval_solver.Y = torch.zeros(P, N + 1, equation.dim_y, device=device)
eval_solver.Z = torch.zeros(P, N,     equation.dim_y, equation.dim_w, device=device)
eval_solver.Y[:, -1, :] = equation.terminal_condition(eval_solver.X[:, -1, :])

# Step 3: single evaluation pass — no gradient updates, just network forward passes
Y_eval_t, _ = eval_solver._update_iterates(eval_solver.Y, eval_solver.Z)

# Extract as numpy: [P, N+1]
t_grid_np = eval_solver.time_grid.cpu().numpy()
Y_eval    = Y_eval_t.cpu().numpy()[:, :, 0]   # [P, N+1]

# Discount factors and exposure profiles
discount  = np.exp(-r * t_grid_np)
epe = np.mean(discount * np.maximum(Y_eval, 0), axis=0)
ene = np.mean(discount * np.minimum(Y_eval, 0), axis=0)

# Exact DEPE: the discounted expected payoff E[e^{-rT} max(X_T-K,0)] = V_BS(0, S0)
# is a Q-martingale, so it is constant at the t=0 price for all t. ENE=0 since call >= 0.
epe_exact = np.full_like(t_grid_np, exact_price)
ene_exact = np.zeros_like(t_grid_np)

print(f'Done. DEPE at t=0: ${epe[0]:.4f}  (BS: ${epe_exact[0]:.4f})')

## 11. Plot Exposure Profiles

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(t_grid_np, epe_exact, 'b--', linewidth=2.5, label='DEPE (Black-Scholes)', alpha=0.85)
ax.plot(t_grid_np, epe,       'b-',  linewidth=2,   label='DEPE (Deep Picard)',   alpha=0.95)

ax.plot(t_grid_np, ene_exact, 'r--', linewidth=2.5, label='DENE (Black-Scholes)', alpha=0.85)
ax.plot(t_grid_np, ene,       'r-',  linewidth=2,   label='DENE (Deep Picard)',   alpha=0.95)

ax.set_xlabel('Time (years)', fontsize=13)
ax.set_ylabel('Discounted Exposure ($)', fontsize=13)
ax.set_title(
    f'Expected Positive and Negative Exposure Profiles\n'
    f'Call Option (K=${K:.0f}, S0=${S0:.0f}, σ={sigma*100:.0f}%, r={r*100:.0f}%)',
    fontsize=14, fontweight='bold'
)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Statistical Analysis of Simulated Paths

In [ ]:
terminal_values = Y_eval[:, -1]   # Y_T = payoff at maturity

print('Statistics at Maturity (T=1):')
print(f'  Mean:   ${np.mean(terminal_values):.4f}')
print(f'  Std:    ${np.std(terminal_values):.4f}')
print(f'  Min:    ${np.min(terminal_values):.4f}')
print(f'  Max:    ${np.max(terminal_values):.4f}')

for p in [5, 25, 50, 75, 95]:
    print(f'  {p}th pct: ${np.percentile(terminal_values, p):.4f}')

## 13. Distribution of Terminal Option Values

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(terminal_values, bins=50, density=True, alpha=0.7,
        color='steelblue', edgecolor='black', linewidth=0.5)

ax.axvline(exact_price, color='red',   linestyle='--', linewidth=2,
           label=f'Black-Scholes: ${exact_price:.2f}')
ax.axvline(np.mean(terminal_values), color='green', linestyle='-', linewidth=2,
           label=f'Deep Picard mean: ${np.mean(terminal_values):.2f}')

ax.set_xlabel('Option Value at Maturity ($)', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Distribution of Terminal Option Values', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 14. Summary Table

In [ ]:
df_exposure = pd.DataFrame({
    'Time':             t_grid_np,
    'DEPE_Exact':       epe_exact,
    'DEPE_DeepPicard':  epe,
    'DENE_Exact':       ene_exact,
    'DENE_DeepPicard':  ene,
})

print(f"\n{'='*70}")
print('MODEL SUMMARY')
print(f"{'='*70}")
print(f'  Equation:            EuropeanCallEquation')
print(f'  Solver:              UncoupledFBSDESolver (Deep Picard Iteration)')
print(f'  Z Method:            gradient (automatic differentiation)')
print(f'  S0={S0}, K={K}, r={r*100}%, sigma={sigma*100}%, T={T}')
print(f'  Time steps N={N}, Training paths={num_paths}, Picard iters={picard_iters}')
print(f'  Network:             {nn_Y}')
print(f'  Parameters:          {sum(p.numel() for p in nn_Y.parameters()):,}')
print(f'  Black-Scholes Price: ${exact_price:.6f}')
print(f'  Deep Picard Y_0:     ${deep_price:.6f}')
print(f'  Absolute Error:      ${error:.6f}')
print(f'  Relative Error:      {rel_error:.4f}%')
print(f'  Training time:       {elapsed:.1f}s')
print(f"{'='*70}")

print('\nExposure Profile (first 10 rows):')
display(df_exposure.head(10))

## 15. Export Results (Optional)

In [ ]:
# Uncomment to save
# df_exposure.to_excel('call_option_deep_picard_results.xlsx', index=False)
# print("Saved to 'call_option_deep_picard_results.xlsx'")

## 16. Spot Grid: Deep Picard Pricing Error vs Black-Scholes

We sweep a grid of initial spot prices $S_0$ (all other parameters fixed) and for each compute $Y_0$ via Deep Picard. Errors are normalised by the **at-the-money Vega**:

$$\text{Vega}_{\text{ATM}} = K \cdot N'(d_1^{\text{ATM}}) \cdot \sqrt{T}, \qquad d_1^{\text{ATM}} = \frac{(r + \frac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}$$

so the normalised error is dimensionless and scale-invariant across the moneyness spectrum.

In [ ]:
# ── Spot grid ─────────────────────────────────────────────────────────────────
spot_grid = np.linspace(70, 130, 25)   # 25 spots from 70% to 130% of K

# ── ATM Vega (fixed normaliser across all spots) ──────────────────────────────
d1_atm   = (r + 0.5 * sigma**2) * T / (sigma * np.sqrt(T))
atm_vega = K * norm.pdf(d1_atm) * np.sqrt(T)
print(f'ATM Vega (normaliser): {atm_vega:.4f}')

# ── Per-spot solver config ────────────────────────────────────────────────────
solver_cfg_grid = SolverConfig(
    T=T, N=N, num_paths=num_paths,
    picard_iterations=picard_iters,
    z_method='gradient',
    device=device
)

results_grid = []

for s0_i in spot_grid:
    bs_price_i  = bs_call(s0_i, K, r, sigma, T)
    moneyness_i = s0_i / K

    eq_i = EuropeanCallEquation(S0=s0_i, K=K, r=r, sigma=sigma, T=T, device=device)

    torch.manual_seed(42)
    nn_Y_i   = MLP(input_dim=1 + eq_i.dim_x, output_dim=eq_i.dim_y, hidden_dims=hidden_dims)
    solver_i = UncoupledFBSDESolver(eq_i, solver_cfg_grid, train_cfg, nn_Y=nn_Y_i, nn_Z=None)

    sol_i     = solver_i.solve()
    deep_y0_i = float(np.mean(sol_i['Y'][:, 0, 0]))
    abs_err_i = deep_y0_i - bs_price_i

    results_grid.append({
        'S0':         s0_i,
        'Moneyness':  moneyness_i,
        'BS_Price':   bs_price_i,
        'Deep_Y0':    deep_y0_i,
        'Abs_Error':  abs_err_i,
        'Norm_Error': abs_err_i / atm_vega,
    })
    print(f'  S0={s0_i:6.1f} | m={moneyness_i:.3f} | BS={bs_price_i:.4f} '
          f'| Y0={deep_y0_i:.4f} | err={abs_err_i:+.4f} | norm={abs_err_i/atm_vega:+.4f}')

df_grid = pd.DataFrame(results_grid)
df_grid['Bucket'] = pd.cut(
    df_grid['Moneyness'],
    bins=[0, 0.90, 0.97, 1.03, 1.10, 99],
    labels=['Deep OTM', 'OTM', 'ATM', 'ITM', 'Deep ITM']
)
print(f'\nGrid complete — {len(df_grid)} spots evaluated.')

### 16a. Summary Statistics

In [ ]:
stats = {
    'Mean error (raw $)':       df_grid['Abs_Error'].mean(),
    'Mean |error| (raw $)':     df_grid['Abs_Error'].abs().mean(),
    'Max |error| (raw $)':      df_grid['Abs_Error'].abs().max(),
    'Mean norm error':          df_grid['Norm_Error'].mean(),
    'Mean |norm error|':        df_grid['Norm_Error'].abs().mean(),
    'Max |norm error|':         df_grid['Norm_Error'].abs().max(),
    'Std norm error':           df_grid['Norm_Error'].std(),
    'ATM Vega (normaliser)':    atm_vega,
}
print(f"{'='*52}")
print('NORMALISED PRICING ERROR - SUMMARY STATISTICS')
print(f"{'='*52}")
for k, v in stats.items():
    print(f'  {k:<35} {v:+.6f}')
print(f"{'='*52}")

print('\nMean normalised error by Moneyness Bucket:')
display(
    df_grid.groupby('Bucket', observed=True)['Norm_Error']
           .agg(Mean='mean', Std='std', MaxAbs=lambda x: x.abs().max())
           .round(5)
)

### 16b. Scatter Plot: Normalised Error by Moneyness

In [ ]:
bucket_colors = {
    'Deep OTM': '#e74c3c',
    'OTM':      '#e67e22',
    'ATM':      '#2ecc71',
    'ITM':      '#3498db',
    'Deep ITM': '#9b59b6',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: normalised error vs moneyness
ax = axes[0]
for bucket, grp in df_grid.groupby('Bucket', observed=True):
    ax.scatter(grp['Moneyness'], grp['Norm_Error'],
               color=bucket_colors[bucket], s=90, zorder=3,
               edgecolors='k', linewidths=0.5, label=bucket)
ax.axhline(0,   color='black', linewidth=1.2, linestyle='--', alpha=0.6)
ax.axvline(1.0, color='grey',  linewidth=1.0, linestyle=':',  alpha=0.7)
ax.axvspan(0.97, 1.03, alpha=0.07, color='green')
ax.set_xlabel('Moneyness  S0 / K', fontsize=12)
ax.set_ylabel('Normalised Error  (err / ATM Vega)', fontsize=12)
ax.set_title('Normalised Pricing Error by Moneyness', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: absolute error vs BS price
ax2 = axes[1]
for bucket, grp in df_grid.groupby('Bucket', observed=True):
    ax2.scatter(grp['BS_Price'], grp['Abs_Error'],
                color=bucket_colors[bucket], s=90, zorder=3,
                edgecolors='k', linewidths=0.5, label=bucket)
ax2.axhline(0, color='black', linewidth=1.2, linestyle='--', alpha=0.6)
ax2.set_xlabel('Black-Scholes Price ($)', fontsize=12)
ax2.set_ylabel('Absolute Error  Y0_hat - V_BS  ($)', fontsize=12)
ax2.set_title('Absolute Error vs Black-Scholes Price', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle(
    f'Deep Picard vs Black-Scholes  |  K={K}, sigma={sigma*100:.0f}%, r={r*100:.0f}%, T={T}y  '
    f'|  ATM Vega = {atm_vega:.3f}',
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()

### 16c. Full Results Table

In [ ]:
display(
    df_grid[['S0','Moneyness','Bucket','BS_Price','Deep_Y0','Abs_Error','Norm_Error']]
    .style
    .format({
        'S0':         '{:.1f}',
        'Moneyness':  '{:.3f}',
        'BS_Price':   '${:.4f}',
        'Deep_Y0':    '${:.4f}',
        'Abs_Error':  '{:+.4f}',
        'Norm_Error': '{:+.5f}',
    })
    .background_gradient(subset=['Norm_Error'], cmap='RdYlGn_r', vmin=-0.05, vmax=0.05)
    .set_caption('Pricing Error Grid - Normalised by ATM Vega')
)

## Conclusion

This notebook replaced the original Deep BSDE solver (Han, Jentzen, E 2018) with the
**Deep Picard Iteration** from the `dim_fbsde` library. Key differences:

| | Original Deep BSDE | Deep Picard (this notebook) |
|---|---|---|
| **Equation class** | Custom `CallOption` / `Equation` | `EuropeanCallEquation(FBSDE)` |
| **Solver** | `BSDESolver` / `NonsharedModel` | `UncoupledFBSDESolver` |
| **Z method** | Learned Z₀ + per-step subnets | Automatic differentiation (gradient) |
| **Iteration** | Single forward pass, learn Y₀ globally | Picard fixed-point on full path |
| **Driver** | $f = -rY_t$ (hard-coded) | $f = -rY_t$ (via `EuropeanCallEquation.driver`) |

### References
- Han, J., Jentzen, A., & E, W. (2018). Solving high-dimensional PDEs using deep learning. PNAS.
- Husain, B. S. (2025). Deep Learning for High-Dimensional FBSDEs. University of Toronto.
- Gnoatto, A., Picarelli, A., & Reisinger, C. (2020). Deep xVA solver. arXiv:2005.02633.